# Generative AI Unit End Project: Creating Designs with OpenAI and Gradio

## Project Overview

In this project, build a simple web-based creative design tool that generates an image from a user's text prompt. The application should connect an OpenAI image-generation model to a Gradio interface so a user can describe a banner, poster, or other visual concept and view the generated result.

## Project Context

Imagine that you are a designer at a creative agency developing promotional concepts for a high-profile entertainment client such as Netflix. Your goal is to create a lightweight platform that transforms written creative directions into generated visual concepts.

## 1. Environment Setup

In [1]:
# Run this cell only if the required packages are not already installed.
# %pip install -qU openai gradio pillow requests python-dotenv

In [2]:
import os
from PIL import Image
import base64
import io
import gradio as gr
from openai import OpenAI, APIConnectionError, AuthenticationError, BadRequestError, OpenAIError, RateLimitError

## 2. Configure OpenAI API Access


In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_BASE_URL"] = "https://openai.vocareum.com/v1"
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


In [4]:
client = OpenAI()

## 3. Plan the Image-Generation Request

In [12]:
prompt = 'lunar eclipse'

# Send request to OpenAI
result = client.images.generate(
    model="gpt-image-1-mini",
    prompt=prompt,
    size='1024x1024',
    quality='low'
)

# Extract the image bytes
image_base64 = result.data[0].b64_json
image_bytes = base64.b64decode(image_base64)

# Open the image
Image.open(io.BytesIO(image_bytes)).show()

This is the basic code for requesting an image from OpenAI's API. I used 'gpt-image-1-mini' because DALL-E versions are no longer in service.

## 4. Create the `generate_image` Function

In [6]:
def generate_image(prompt: str, img_type: str = 'Regular', shape: str = 'Square', amount: int = 1) -> list[Image.Image]:
    prompt = prompt.strip()
    if not prompt:
        raise gr.Error("Please enter an image description.")

    if img_type == 'Marketing':
        prompt = f"Make marketing image from: {prompt}"

    match shape:
        case 'Square':
            size = '1024x1024'
        case 'Landscape':
            size = '1536x1024'
        case 'Portrait':
            size = '1024x1536'

    try:
        # This is where the API call is made. If successful, the image data will be in 'result'
        result = client.images.generate(
            model="gpt-image-1-mini",
            prompt=prompt,  # The user's prompt gets passed to the API here
            size=size,
            quality='low',
            n=amount
        )

    except AuthenticationError:
        raise gr.Error("The OpenAI API key is invalid or expired.")

    except RateLimitError:
        raise gr.Error("API limit reached. Please wait and try again.")

    except APIConnectionError:
        raise gr.Error("Could not connect to OpenAI. Check your internet, VPN, or proxy.")

    except BadRequestError as error:
        raise gr.Error("Your prompt may not meet safety guidelines. Try submitting again or use a different prompt.")

    except OpenAIError:
        raise gr.Error("OpenAI could not generate the image. Try again with a different prompt.")

    except Exception:
        raise gr.Error("Sorry, an unexpected error occurred. Try submitting again or use a different prompt.")

    # If the API successfully returned an image(s), it gets converted here
    images = []
    try:
        for i in range(amount):
            # OpenAI returns the image as base64 text
            image_base64 = result.data[i].b64_json

            # Decode it into bytes
            image_bytes = base64.b64decode(image_base64)

            # Wrap bytes with io.BytesIO so PIL can read it like a file
            images.append(Image.open(io.BytesIO(image_bytes)))

    except Exception:
        raise gr.Error("Sorry, the image couldn't be processed. Please try again.")

    # The function the returns the images(s) for Gradio to display here
    return images


## 5. Test Image Generation in the Notebook


In [7]:
prompt1 = """
• Subject or focal point: avatar ice person (light blue icy skin, light blue hair, holding ice weapons) and their family
• Visual style: realistic, epic
• Composition: symmetrical framing, centered
• Mood: awe inspiring, ambitious
• Lighting: early dawn
It's a movie poster for "Avatar: Snowflake and Ice" """

prompt2 = """
• Subject or focal point: fast and furious cast
• Visual style: photorealistic, cinematic
• Composition: symmetrical framing, centered, racing cars behind them
• Mood: intense, suspense, fearless
• Lighting: Dramatic sunset
It's a movie poster for "Fast & Furious: Ultimate Race" """

tests = [
    {
        'prompt': prompt1,
        'img_type': 'Marketing',
        'shape': 'Portrait',
        'amount': 1
    },
    {
        'prompt': prompt2,
        'img_type': 'Marketing',
        'shape': 'Landscape',
        'amount': 1
    },
]

for test in tests:
    imgs = generate_image(**test)

    for img in imgs:
        img.show()

## 6. Build the Gradio Interface

In [18]:
info = """A useful prompt may describe elements such as:
• Subject or focal point
• Visual style
• Composition
• Mood
• Lighting
• Color direction
• Background
• Intended format, such as a promotional poster or banner"""

demo = gr.Interface(
    # Function
    fn=generate_image,

    # Inputs
    inputs=[
        # Gradio-provided textbox for user to input a prompt
        gr.Textbox(
            label='Image description',
            info=info,
            autofocus=True,
            max_length=1000,
            lines=3,
            max_lines=6
        ),
        gr.Dropdown(
            label='Image type',
            choices=['Regular', 'Marketing'],
            value="Regular"
        ),
        gr.Dropdown(
            label='Shape',
            choices=['Square', 'Landscape', 'Portrait'],
            value="Square"

        ),
        gr.Number(
            label="Number of Images",
            value=4,
            minimum=1,
            maximum=8,
            precision=0,
        )
    ],

    # Outputs
    outputs=gr.Gallery(
        label="Generated Images",
        columns=2,          # Number of columns for the grid thumbnail view
        object_fit="contain", # Preserves aspect ratio during arrow preview
        height="auto"
    ),

    title="Visual Content Generator",
    description="Rapidly prototype visual design mockups."
)

## 7. Launch and Test the Application


In [ ]:
demo.launch(
    inline=True  # Shows browser within the code cell
)

## 8. Review of the Creative Design Experience


**Image accuracy:**

Based off of my experience, the images always accurately aligned with the prompts. The images may come out blurry and imperfect, but this can be improved by changing the quality value to `medium` or `high` or using a better image model, such as `gpt-image-2` if needed.

**Effect of prompt specificity on image:**

Adding specific detail reduces arbitrary decisions made by the model, which results in content aligned with the user's needs. Some examples are specifying a background, lighting effect, the position of objects on the image, and style of the image. For example, if the user asks for a picture of a dog, the background may be a blank slate. If the user asks for a picture of a dog in a park, it will recreate a park, which may be more suited for a movie poster.

**Interface ease of use**

Gradio provides a very clean, simple, and intuitive interface. It is easy for a new user to know how to submit their prompt and then be able to see the image on the screen.

**Errors and limitations encountered**

The model lacks a way to edit a generated image, it can only start over from scratch.

The entire process can raise a variety of errors, such as BadRequestError and OpenAIError, and some of these errors can have a variety of specific error codes and attributes, so properly handling errors requires extensive testing and investigation.


**Possible improvements**

Brainstorming tools such as a list of common adjectives, styles, moods, light effects, scenes, or full prompt presets could help a user iterate through more ideas quicker. A large gallery of real world examples could also help a user ideate and refine content quicker.

Specialized editing tools such as editing colors, adding filters, and overlaying text would give the user more customization options.

Overall, this is a powerful tool for any type of visual content creation. It can create polished content that lines up with realistic examples, and could drastically reduce the amount of time to create real content.


## Final Summary

- How OpenAI API access was configured
    - An API key obtained from Vocareum was used to set up the OpenAI client object. It needs to be pasted in the cell that asks for it near the beginning.
- How the text prompt was sent to the image-generation model
    - It is sent to OpenAI's API through the `client.images.generate()` function.
- How the image response was processed
    - The API returns a JSON object which contains the image data encoded in base64. A base64 decoder is used to convert it into bytes, which is then passed into an io.BytesIO object which PIL.Image is then able to take into its constructor.
- How `generate_image` connects the API workflow
    - It handles prompt validation, it maps user inputs such as image shape, amount, and type to OpenAI's API requirements, initiates the request, handles potential errors, converts the image to a suitable format, and returns the final image(s).
- How the Gradio `Interface` enables user interaction
    - It handles the entire UI using syntax that is easy and minimal for the programmer to set up, and components that are easy and intuitive for the user to use.
- What you observed while testing the design generator
    - It works well on any prompt that it is given. It handles different aspects like objects, positioning, style, lighting, very well. Text may be unreadable, but these are only mockups for brainstorming, and upgrading the model would reduce that problem.
    - The model will still deny some prompts if they do not meet OpenAI's safety guidelines.
- Any important limitations or potential improvements
    - Text may not be readable, pictures may look imperfect, images can not be edited or tweaked. Potential improvements include more editing and brainstorming tools and a better model.
